In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# 1. Download and Prepare Dataset
!wget https://raw.githubusercontent.com/SamirMoustafa/nmt-with-attention-for-ar-to-en/master/ara_.txt

with open('ara_.txt', 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

# Increased dataset size from 50k to 100k if available
eng_ara_pairs = []
for line in lines[:100000]:  # Use more examples if available
    if '\t' in line:
        eng, ara = line.split('\t')[:2]
        eng_ara_pairs.append([eng, ara])

def preprocess_text(text):
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    text = re.sub(r'[" "]+', " ", text)
    text = re.sub(r"[^a-zA-Z?.!,¿ء-ي]+", " ", text)
    text = text.strip()
    return '<start> ' + text + ' <end>'

word_pairs = [[preprocess_text(eng), preprocess_text(ara)] for eng, ara in eng_ara_pairs]
train_pairs, test_pairs = train_test_split(word_pairs, test_size=0.2, random_state=42)

# 2. Tokenization
eng_tokenizer = keras.preprocessing.text.Tokenizer(filters='')
eng_tokenizer.fit_on_texts([eng for eng, ara in train_pairs])
eng_vocab_size = len(eng_tokenizer.word_index) + 1

ara_tokenizer = keras.preprocessing.text.Tokenizer(filters='')
ara_tokenizer.fit_on_texts([ara for eng, ara in train_pairs])
ara_vocab_size = len(ara_tokenizer.word_index) + 1

def tokenize_pairs(pairs):
    eng_texts = [eng for eng, ara in pairs]
    ara_texts = [ara for eng, ara in pairs]

    eng_sequences = eng_tokenizer.texts_to_sequences(eng_texts)
    ara_sequences = ara_tokenizer.texts_to_sequences(ara_texts)

    eng_sequences = keras.preprocessing.sequence.pad_sequences(eng_sequences, padding='post')
    ara_sequences = keras.preprocessing.sequence.pad_sequences(ara_sequences, padding='post')

    return eng_sequences, ara_sequences

train_eng, train_ara = tokenize_pairs(train_pairs)
test_eng, test_ara = tokenize_pairs(test_pairs)

max_length_eng = train_eng.shape[1]
max_length_ara = train_ara.shape[1]

# 3. Transformer Components
class PositionalEncoding(layers.Layer):
    def __init__(self, max_position, d_model):
        super(PositionalEncoding, self).__init__()
        self.max_position = max_position
        self.d_model = d_model

        position = tf.range(max_position, dtype=tf.float32)[:, tf.newaxis]
        div_term = tf.exp(tf.range(0, d_model, 2, dtype=tf.float32) * (-tf.math.log(10000.0) / d_model))

        pe = tf.zeros((max_position, d_model))
        pe = pe.numpy()
        pe[:, 0::2] = tf.sin(position * div_term).numpy()
        pe[:, 1::2] = tf.cos(position * div_term).numpy()
        pe = pe[tf.newaxis, ...]

        self.pe = tf.convert_to_tensor(pe, dtype=tf.float32)

    def call(self, inputs):
        if isinstance(inputs, tf.SparseTensor):
            inputs = tf.sparse.to_dense(inputs)
        seq_len = tf.shape(inputs)[1]
        return inputs + self.pe[:, :seq_len, :]

    def compute_output_shape(self, input_shape):
        return input_shape

def scaled_dot_product_attention(query, key, value, mask):
    matmul_qk = tf.matmul(query, key, transpose_b=True)
    depth = tf.cast(tf.shape(key)[-1], tf.float32)
    logits = matmul_qk / tf.math.sqrt(depth)

    if mask is not None:
        logits += (mask * -1e9)

    attention_weights = tf.nn.softmax(logits, axis=-1)
    output = tf.matmul(attention_weights, value)
    return output, attention_weights

class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads, name="multi_head_attention"):
        super(MultiHeadAttention, self).__init__(name=name)
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        # Added L2 regularization to all Dense layers
        self.query_dense = layers.Dense(units=d_model, kernel_regularizer=keras.regularizers.l2(0.0005))
        self.key_dense = layers.Dense(units=d_model, kernel_regularizer=keras.regularizers.l2(0.0005))
        self.value_dense = layers.Dense(units=d_model, kernel_regularizer=keras.regularizers.l2(0.0005))
        self.dense = layers.Dense(units=d_model, kernel_regularizer=keras.regularizers.l2(0.0005))

    def split_heads(self, inputs, batch_size):
        inputs = tf.reshape(inputs, shape=(batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(inputs, perm=[0, 2, 1, 3])

    def call(self, inputs):
        query, key, value, mask = inputs['query'], inputs['key'], inputs['value'], inputs['mask']
        batch_size = tf.shape(query)[0]

        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        scaled_attention, attention_weights = scaled_dot_product_attention(query, key, value, mask)

        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))
        output = self.dense(concat_attention)
        return output, attention_weights

def encoder_layer(units, d_model, num_heads, dropout, name="encoder_layer"):
    inputs = keras.Input(shape=(None, d_model), name="inputs")
    padding_mask = keras.Input(shape=(1, 1, None), name="padding_mask")

    attention, _ = MultiHeadAttention(d_model, num_heads, name="attention")({
        'query': inputs, 'key': inputs, 'value': inputs, 'mask': padding_mask})

    # Increased dropout rate
    attention = layers.Dropout(dropout)(attention)
    attention = layers.LayerNormalization(epsilon=1e-6)(inputs + attention)

    # Added L2 regularization to Dense layers
    outputs = layers.Dense(
        units=units,
        activation='relu',
        kernel_regularizer=keras.regularizers.l2(0.001)
    )(attention)

    outputs = layers.Dense(
        units=d_model,
        kernel_regularizer=keras.regularizers.l2(0.001)
    )(outputs)

    outputs = layers.Dropout(dropout)(outputs)
    outputs = layers.LayerNormalization(epsilon=1e-6)(attention + outputs)

    return keras.Model(inputs=[inputs, padding_mask], outputs=outputs, name=name)

def decoder_layer(units, d_model, num_heads, dropout, name="decoder_layer"):
    inputs = keras.Input(shape=(None, d_model), name="inputs")
    enc_outputs = keras.Input(shape=(None, d_model), name="encoder_outputs")
    look_ahead_mask = keras.Input(shape=(1, None, None), name="look_ahead_mask")
    padding_mask = keras.Input(shape=(1, 1, None), name='padding_mask')

    attention1, attn_weights1 = MultiHeadAttention(d_model, num_heads, name="attention_1")({
        'query': inputs, 'key': inputs, 'value': inputs, 'mask': look_ahead_mask})

    # Increased dropout
    attention1 = layers.Dropout(dropout)(attention1)
    attention1 = layers.LayerNormalization(epsilon=1e-6)(attention1 + inputs)

    attention2, attn_weights2 = MultiHeadAttention(d_model, num_heads, name="attention_2")({
        'query': attention1, 'key': enc_outputs, 'value': enc_outputs, 'mask': padding_mask})

    attention2 = layers.Dropout(dropout)(attention2)
    attention2 = layers.LayerNormalization(epsilon=1e-6)(attention2 + attention1)

    # Added L2 regularization to Dense layers
    outputs = layers.Dense(
        units=units,
        activation='relu',
        kernel_regularizer=keras.regularizers.l2(0.001)
    )(attention2)

    outputs = layers.Dense(
        units=d_model,
        kernel_regularizer=keras.regularizers.l2(0.001)
    )(outputs)

    outputs = layers.Dropout(dropout)(outputs)
    outputs = layers.LayerNormalization(epsilon=1e-6)(outputs + attention2)

    return keras.Model(
        inputs=[inputs, enc_outputs, look_ahead_mask, padding_mask],
        outputs=[outputs, attn_weights1, attn_weights2],
        name=name)

def encoder(vocab_size, num_layers, units, d_model, num_heads, dropout, name="encoder"):
    inputs = keras.Input(shape=(None,), name="inputs")
    padding_mask = keras.Input(shape=(1, 1, None), name="padding_mask")

    embeddings = layers.Embedding(vocab_size, d_model)(inputs)
    embeddings *= tf.math.sqrt(tf.cast(d_model, tf.float32))
    embeddings = PositionalEncoding(max_length_ara, d_model)(embeddings)
    outputs = layers.Dropout(dropout)(embeddings)

    for i in range(num_layers):
        outputs = encoder_layer(
            units=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            name=f"encoder_layer_{i}"
        )([outputs, padding_mask])

    return keras.Model(inputs=[inputs, padding_mask], outputs=outputs, name=name)

def decoder(vocab_size, num_layers, units, d_model, num_heads, dropout, name="decoder"):
    inputs = keras.Input(shape=(None,), name="inputs")
    enc_outputs = keras.Input(shape=(None, d_model), name="encoder_outputs")
    look_ahead_mask = keras.Input(shape=(1, None, None), name="look_ahead_mask")
    padding_mask = keras.Input(shape=(1, 1, None), name='padding_mask')

    embeddings = layers.Embedding(vocab_size, d_model)(inputs)
    embeddings *= tf.math.sqrt(tf.cast(d_model, tf.float32))
    embeddings = PositionalEncoding(max_length_eng, d_model)(embeddings)
    outputs = layers.Dropout(dropout)(embeddings)

    attention_weights = {}
    for i in range(num_layers):
        outputs, block1, block2 = decoder_layer(
            units=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            name=f"decoder_layer_{i}"
        )([outputs, enc_outputs, look_ahead_mask, padding_mask])

        attention_weights[f'decoder_layer{i+1}_block1'] = block1
        attention_weights[f'decoder_layer{i+1}_block2'] = block2

    return keras.Model(
        inputs=[inputs, enc_outputs, look_ahead_mask, padding_mask],
        outputs=[outputs, attention_weights],
        name=name)

# 4. Transformer Model
def transformer(input_vocab_size, target_vocab_size, num_layers, units, d_model, num_heads, dropout, name="transformer"):
    inputs = keras.Input(shape=(None,), name="inputs")
    dec_inputs = keras.Input(shape=(None,), name="dec_inputs")

    enc_padding_mask = layers.Lambda(
        lambda x: tf.cast(tf.math.equal(x, 0), tf.float32)[:, tf.newaxis, tf.newaxis, :],
        name='enc_padding_mask')(inputs)

    def create_look_ahead_mask(size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask[tf.newaxis, tf.newaxis, :, :]

    look_ahead_mask = layers.Lambda(
        lambda x: create_look_ahead_mask(tf.shape(x)[1]),
        name='look_ahead_mask')(dec_inputs)

    dec_padding_mask = layers.Lambda(
        lambda x: tf.cast(tf.math.equal(x, 0), tf.float32)[:, tf.newaxis, tf.newaxis, :],
        name='dec_padding_mask')(inputs)

    enc_outputs = encoder(
        vocab_size=input_vocab_size,
        num_layers=num_layers,
        units=units,
        d_model=d_model,
        num_heads=num_heads,
        dropout=dropout,
    )([inputs, enc_padding_mask])

    dec_outputs, _ = decoder(
        vocab_size=target_vocab_size,
        num_layers=num_layers,
        units=units,
        d_model=d_model,
        num_heads=num_heads,
        dropout=dropout,
    )([dec_inputs, enc_outputs, look_ahead_mask, dec_padding_mask])

    # Add L2 regularization to final output layer
    outputs = layers.Dense(
        target_vocab_size,
        name="outputs",
        kernel_regularizer=keras.regularizers.l2(0.001)
    )(dec_outputs)

    return keras.Model(inputs=[inputs, dec_inputs], outputs=outputs, name=name)

# 5. Data Augmentation
def augment_data(seq, max_len, noise_factor=0.1):
    # Add random word dropout
    mask = np.random.rand(len(seq)) > noise_factor
    augmented_seq = seq * mask

    # Add slight position shuffling (for words close to each other)
    indices = np.arange(len(seq))
    for i in range(len(indices)-1):
        if np.random.rand() < noise_factor/2:
            indices[i], indices[i+1] = indices[i+1], indices[i]

    augmented_seq = [seq[i] for i in indices]

    # Pad sequence to max_len
    if len(augmented_seq) < max_len:
        augmented_seq = augmented_seq + [0] * (max_len - len(augmented_seq))
    else:
        augmented_seq = augmented_seq[:max_len]

    return augmented_seq

def create_dataset_with_augmentation(ara_sequences, eng_sequences, batch_size):
    augmented_ara = []
    augmented_eng = []

    # Original data
    for ara_seq, eng_seq in zip(ara_sequences, eng_sequences):
        augmented_ara.append(ara_seq)
        augmented_eng.append(eng_seq)

        # Add augmented version
        if np.random.rand() < 0.3:  # 30% chance of augmentation
            augmented_ara.append(augment_data(ara_seq, len(ara_seq)))
            augmented_eng.append(eng_seq)  # keep target sequence the same

    augmented_ara = np.array(augmented_ara)
    augmented_eng = np.array(augmented_eng)

    decoder_input = augmented_eng[:, :-1]
    decoder_target = augmented_eng[:, 1:]

    dataset = tf.data.Dataset.from_tensor_slices((
        {'inputs': augmented_ara, 'dec_inputs': decoder_input},
        decoder_target
    ))

    dataset = dataset.cache()
    dataset = dataset.shuffle(buffer_size=len(augmented_ara))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# 6. Training Setup - Reduced model complexity
NUM_LAYERS = 2  # Reduced from 4
D_MODEL = 64    # Reduced from 128
NUM_HEADS = 4   # Reduced from 8
UNITS = 256     # Reduced from 512
DROPOUT = 0.3   # Increased from 0.1
EPOCHS = 100
BATCH_SIZE = 32  # Reduced from 64

def loss_function(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')(y_true, y_pred)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_accuracy(y_true, y_pred):
    y_pred = tf.argmax(y_pred, axis=-1)
    y_pred = tf.cast(y_pred, y_true.dtype)

    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    correct = tf.cast(tf.equal(y_true, y_pred), tf.float32)
    correct *= mask

    return tf.reduce_sum(correct) / tf.reduce_sum(mask)

# Modified learning rate schedule with decay
class CustomSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps
        # Add decay factor
        self.decay_factor = 0.98

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)
        # Apply exponential decay
        decay = self.decay_factor ** (step / 1000.0)
        return lr * decay

learning_rate = CustomSchedule(D_MODEL)
# Added gradient clipping
optimizer = keras.optimizers.Adam(
    learning_rate,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=1e-9,
    clipnorm=1.0  # Clip gradients
)

# Create the model
transformer_model = transformer(
    input_vocab_size=ara_vocab_size,
    target_vocab_size=eng_vocab_size,
    num_layers=NUM_LAYERS,
    units=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT)

transformer_model.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=[masked_accuracy]
)

# Create datasets with augmentation
train_dataset = create_dataset_with_augmentation(train_ara, train_eng, BATCH_SIZE)
test_dataset = create_dataset_with_augmentation(test_ara, test_eng, BATCH_SIZE)

# Define translation function for validation callback
def translate(sentence):
    sentence = preprocess_text(sentence)
    inputs = [ara_tokenizer.word_index.get(word, 1) for word in sentence.split(' ')]
    inputs = keras.preprocessing.sequence.pad_sequences([inputs], maxlen=max_length_ara, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    output = tf.expand_dims([eng_tokenizer.word_index['<start>']], 0)

    for i in range(max_length_eng):
        enc_padding_mask = tf.cast(tf.math.equal(inputs, 0), tf.float32)[:, tf.newaxis, tf.newaxis, :]
        look_ahead_mask = 1 - tf.linalg.band_part(tf.ones((tf.shape(output)[1], tf.shape(output)[1])), -1, 0)
        look_ahead_mask = look_ahead_mask[tf.newaxis, tf.newaxis, :, :]
        dec_padding_mask = tf.cast(tf.math.equal(inputs, 0), tf.float32)[:, tf.newaxis, tf.newaxis, :]

        predictions = transformer_model({'inputs': inputs, 'dec_inputs': output}, training=False)

        predictions = predictions[:, -1:, :]
        predicted_id = tf.cast(tf.argmax(predictions, axis=-1), tf.int32)

        if predicted_id == eng_tokenizer.word_index['<end>']:
            break

        output = tf.concat([output, predicted_id], axis=-1)

    predicted_sentence = eng_tokenizer.sequences_to_texts(output.numpy())[0]
    predicted_sentence = predicted_sentence.replace('<start>', '').replace('<end>', '').strip()
    return predicted_sentence

# Add validation callback
test_sentences = [
    "مرحبًا كيف حالك؟",
    "ما هو اسمك؟",
    "هل تتكلم الإنجليزية؟",
    "الطقس جميل اليوم",
    "كيف يمككني مساعدتك"
]

class ValidationCallback(keras.callbacks.Callback):
    def __init__(self, test_sentences):
        self.test_sentences = test_sentences

    def on_epoch_end(self, epoch, logs=None):
        if epoch % 5 == 0:  # Check every 5 epochs
            print(f"\nEpoch {epoch} - Sample translations:")
            for sent in self.test_sentences[:3]:  # Test first 3 sentences
                translation = translate(sent)
                print(f"Arabic: {sent}")
                print(f"English: {translation}")

# Add early stopping
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

validation_callback = ValidationCallback(test_sentences)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    'best_transformer_model.weights.h5',  # Fixed filename
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True
)

callbacks = [early_stopping, validation_callback, model_checkpoint]

# 7. Training
print("Starting training...")
history = transformer_model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=callbacks
)

# 8. Load best model weights
transformer_model.load_weights('best_transformer_model.weights.h5')  # Updated filename

# 9. Plot training history
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['masked_accuracy'], label='Training Accuracy')
plt.plot(history.history['val_masked_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Print final metrics
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
final_train_acc = history.history['masked_accuracy'][-1]
final_val_acc = history.history['val_masked_accuracy'][-1]

print("\nTraining Complete!")
print(f"Final Training Loss: {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")
print(f"Final Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")

# 10. Example translations
print("\nSample Translations:")
for sent in test_sentences:
    translation = translate(sent)
    print(f"Arabic: {sent}")
    print(f"English: {translation}\n")